In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [2]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [3]:
df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\combine_emotion_data.csv")

N_MELS = 128
SR = 22050
DURATION = 3
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 7

print(f'Total clips: {len(df)}')
print(f'Classes: {df["emotion"].nunique()}')

Total clips: 4048
Classes: 7


In [4]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])

class EmoWaveDataset(Dataset):
    def __init__(self, df, augment_data=False):
        self.df = df.reset_index(drop=True)
        self.augment_data = augment_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                           n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)

        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(row['label'], dtype=torch.long)

        if self.augment_data:
            y = self.augment(y, sr)

        return tensor, label

    def augment(self, y, sr):
    # Add random noise
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise

    # Time shift
        shift = np.random.randint(sr // 4)
        y = np.roll(y, shift)

    # Pitch shift
        steps = np.random.uniform(-2, 2)
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

        return y

print('Dataset class defined')

Dataset class defined


In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = EmoWaveDataset(train_df, augment_data=True)
test_dataset = EmoWaveDataset(test_df, augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# verify one batch
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 3238
Test samples:  810
Train batches: 102
Test batches:  26
Batch X shape: torch.Size([32, 1, 128, 130])
Batch y shape: torch.Size([32])


In [6]:
class EmoWaveNet(nn.Module):
    def __init__(self, num_classes=8):
        super(EmoWaveNet, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.classifier(x)
        return x

model = EmoWaveNet(num_classes=NUM_CLASSES).to(device)

In [7]:
dummy = torch.zeros(1, 1, 128, 130).to(device)
with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 130).to(device)
    out = model.conv_block1(dummy)
    out = model.conv_block2(out)
    out = model.conv_block3(out)
    print(out.shape)
    print(out.view(1, -1).shape[1])

torch.Size([1, 128, 16, 16])
32768


In [8]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [9]:
# Replace final layer for 8 classes
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

# Convert grayscale (1 channel) to 3 channels EfficientNet expects
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Move to GPU
efficientnet = efficientnet.to(device)

# Freeze all layers except classifier
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

# Unfreeze first conv too since we changed it
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet ready')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet ready
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=7, bias=True)
)


In [10]:
# 1. criterion defined
criterion = nn.CrossEntropyLoss()

# 2. EfficientNet loaded and modified
import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)
efficientnet = efficientnet.to(device)

# 3. Frozen correctly
for param in efficientnet.parameters():
    param.requires_grad = False
for param in efficientnet.classifier.parameters():
    param.requires_grad = True
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

In [11]:
optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_eff = optim.lr_scheduler.StepLR(optimizer_eff, step_size=10, gamma=0.5)

EPOCHS_EFF = 30
eff_losses = []
eff_accuracies = []

for epoch in range(EPOCHS_EFF):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff.step()
        running_loss += loss.item()

    scheduler_eff.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    eff_losses.append(avg_loss)
    eff_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS_EFF}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 1.4762 | Test Acc: 0.6667
Epoch [2/30] Loss: 1.0716 | Test Acc: 0.6975
Epoch [3/30] Loss: 0.9183 | Test Acc: 0.7173
Epoch [4/30] Loss: 0.8685 | Test Acc: 0.7383
Epoch [5/30] Loss: 0.8158 | Test Acc: 0.7605
Epoch [6/30] Loss: 0.7501 | Test Acc: 0.7605
Epoch [7/30] Loss: 0.7304 | Test Acc: 0.7506
Epoch [8/30] Loss: 0.7201 | Test Acc: 0.7358
Epoch [9/30] Loss: 0.7026 | Test Acc: 0.7753
Epoch [10/30] Loss: 0.6959 | Test Acc: 0.7778
Epoch [11/30] Loss: 0.6413 | Test Acc: 0.7901
Epoch [12/30] Loss: 0.6256 | Test Acc: 0.7790
Epoch [13/30] Loss: 0.6173 | Test Acc: 0.7691
Epoch [14/30] Loss: 0.6123 | Test Acc: 0.7963
Epoch [15/30] Loss: 0.6184 | Test Acc: 0.7802
Epoch [16/30] Loss: 0.5866 | Test Acc: 0.7630
Epoch [17/30] Loss: 0.5987 | Test Acc: 0.7877
Epoch [18/30] Loss: 0.6041 | Test Acc: 0.7840
Epoch [19/30] Loss: 0.5891 | Test Acc: 0.7802
Epoch [20/30] Loss: 0.5915 | Test Acc: 0.7901
Epoch [21/30] Loss: 0.5684 | Test Acc: 0.7889
Epoch [22/30] Loss: 0.5790 | Test Acc: 0.79

In [12]:
# Unfreeze everything
for param in efficientnet.parameters():
    param.requires_grad = True

# Lower learning rate for fine-tuning
optimizer_eff2 = optim.Adam(efficientnet.parameters(), lr=0.0001)
scheduler_eff2 = optim.lr_scheduler.StepLR(optimizer_eff2, step_size=10, gamma=0.5)

EPOCHS_EFF2 = 30
for epoch in range(EPOCHS_EFF2):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff2.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff2.step()
        running_loss += loss.item()

    scheduler_eff2.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch + 31}/{60}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')


Epoch [31/60] Loss: 0.4938 | Test Acc: 0.8346
Epoch [32/60] Loss: 0.2888 | Test Acc: 0.8654
Epoch [33/60] Loss: 0.1867 | Test Acc: 0.8580
Epoch [34/60] Loss: 0.1355 | Test Acc: 0.8667
Epoch [35/60] Loss: 0.1078 | Test Acc: 0.8667
Epoch [36/60] Loss: 0.0802 | Test Acc: 0.8630
Epoch [37/60] Loss: 0.0832 | Test Acc: 0.8654
Epoch [38/60] Loss: 0.0539 | Test Acc: 0.8765
Epoch [39/60] Loss: 0.0581 | Test Acc: 0.8778
Epoch [40/60] Loss: 0.0425 | Test Acc: 0.8864
Epoch [41/60] Loss: 0.0401 | Test Acc: 0.8852
Epoch [42/60] Loss: 0.0297 | Test Acc: 0.8840
Epoch [43/60] Loss: 0.0395 | Test Acc: 0.8852
Epoch [44/60] Loss: 0.0227 | Test Acc: 0.8864
Epoch [45/60] Loss: 0.0267 | Test Acc: 0.8901
Epoch [46/60] Loss: 0.0207 | Test Acc: 0.8914
Epoch [47/60] Loss: 0.0130 | Test Acc: 0.8840
Epoch [48/60] Loss: 0.0145 | Test Acc: 0.8901
Epoch [49/60] Loss: 0.0147 | Test Acc: 0.8901
Epoch [50/60] Loss: 0.0190 | Test Acc: 0.8852
Epoch [51/60] Loss: 0.0199 | Test Acc: 0.8889
Epoch [52/60] Loss: 0.0189 | Test 

In [14]:
torch.save(efficientnet.state_dict(), '../models/emowave_efficientnet1.pth')
print('Model saved')


Model saved
